<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
import json
import pandas as pd
import numpy as np
import torch
from google.colab import drive

# 1. 드라이브 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# 디렉토리 경로 설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model/data'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
EXPORT_DIR = '/content/drive/MyDrive/point-6/ai_model/exported_models'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

# 제약 조건 설정 변수
PHONE_Z_THRESHOLD = 50.0
SPEN_Y_THRESHOLD = 0.8
SPEN_Y_MINUS_THRESHOLD = -0.8
WINDOW_SIZE = 20

# 6개의 가상 드럼 존 레이블 인코딩 매핑 (실제 공간 배치 순서 반영)
LABEL_MAP = {
    'Cymbal1': 0, 'Tom1': 1, 'Cymbal2': 2,
    'Hi-hat': 3, 'Snare': 4, 'Tom2': 5
}

In [15]:
def parse_semicolon_data(data_string):
    """세미콜론으로 구분된 문자열을 실수형 넘파이 배열로 파싱"""
    if pd.isna(data_string):
        return np.array([])
    try:
        return np.array([float(x) for x in str(data_string).split(';') if x.strip() != ''])
    except ValueError:
        return np.array([])

In [16]:
def pad_or_truncate(arr):
    """윈도우 크기에 맞게 패딩을 추가하거나 자르는 함수"""
    if len(arr) >= WINDOW_SIZE:
        return arr[:WINDOW_SIZE]
    else:
        return np.pad(arr, (0, WINDOW_SIZE - len(arr)), 'constant')

In [17]:
def process_raw_files():
    processed_features = []
    processed_labels = []

    # 1. raw 폴더의 모든 CSV 파일 순회
    for filename in os.listdir(RAW_DIR):
        if not filename.endswith('.csv'):
            continue

        filepath = os.path.join(RAW_DIR, filename)
        df = pd.read_csv(filepath)

        # 2. 윈도우 프레임 단위 처리
        for index, row in df.iterrows():
            label_str = str(row['label']).strip()
            if label_str not in LABEL_MAP:
                continue
            label_idx = LABEL_MAP[label_str]

            # 단일 열에서 센서 데이터 파싱
            phone_accel = parse_semicolon_data(row['phoneAccel'])
            phone_gyro = parse_semicolon_data(row['phoneGyro'])
            spen_delta = parse_semicolon_data(row['spenDelta'])

            # 3축 분할 (x, y, z가 번갈아 기록된 구조 분리)
            accel_x = pad_or_truncate(phone_accel[0::3]) if len(phone_accel) > 0 else np.zeros(WINDOW_SIZE)
            accel_y = pad_or_truncate(phone_accel[1::3]) if len(phone_accel) > 0 else np.zeros(WINDOW_SIZE)
            accel_z = pad_or_truncate(phone_accel[2::3]) if len(phone_accel) > 0 else np.zeros(WINDOW_SIZE)

            gyro_x = pad_or_truncate(phone_gyro[0::3]) if len(phone_gyro) > 0 else np.zeros(WINDOW_SIZE)
            gyro_y = pad_or_truncate(phone_gyro[1::3]) if len(phone_gyro) > 0 else np.zeros(WINDOW_SIZE)
            gyro_z = pad_or_truncate(phone_gyro[2::3]) if len(phone_gyro) > 0 else np.zeros(WINDOW_SIZE)

            # 에스펜 2축 분할 (x, y가 번갈아 기록된 구조 분리)
            spen_x = pad_or_truncate(spen_delta[0::2]) if len(spen_delta) > 0 else np.zeros(WINDOW_SIZE)
            spen_y = pad_or_truncate(spen_delta[1::2]) if len(spen_delta) > 0 else np.zeros(WINDOW_SIZE)

            # 3. 다중 피크 처리 규칙 및 방향성 임계값 검사
            max_z_accel = np.max(accel_z) if len(accel_z) > 0 else 0
            max_y_spen = np.max(spen_y) if len(spen_y) > 0 else 0
            min_y_spen = np.min(spen_y) if len(spen_y) > 0 else 0

            # 유효 타격으로 인정
            if max_z_accel >= PHONE_Z_THRESHOLD or max_y_spen >= SPEN_Y_THRESHOLD or min_y_spen <= SPEN_Y_MINUS_THRESHOLD:
                # 8축 데이터를 (8, WINDOW_SIZE) 형태의 행렬로 결합
                feature_matrix = np.vstack([
                    accel_x, accel_y, accel_z,
                    gyro_x, gyro_y, gyro_z,
                    spen_x, spen_y
                ])

                processed_features.append(feature_matrix)
                processed_labels.append(label_idx)

    return np.array(processed_features), np.array(processed_labels)

In [18]:
print("데이터 전처리를 시작합니다...")
features, labels = process_raw_files()

# 데이터 정규화 적용 (평균 0, 분산 1)
mean_vals = np.mean(features, axis=(0, 2), keepdims=True)
std_vals = np.std(features, axis=(0, 2), keepdims=True)
std_vals[std_vals == 0] = 1e-8 # 0으로 나누기 방지

normalized_features = (features - mean_vals) / std_vals

# 안드로이드 실시간 추론을 위한 스케일링 파라미터 저장
scaling_params = {
    "mean": mean_vals.flatten().tolist(),
    "std": std_vals.flatten().tolist()
}
with open(os.path.join(EXPORT_DIR, 'scaling_params.json'), 'w') as f:
    json.dump(scaling_params, f)

# 4. 텐서 형태의 학습용 데이터셋으로 저장
features_tensor = torch.tensor(normalized_features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

torch.save(features_tensor, os.path.join(PROCESSED_DIR, 'features.pt'))
torch.save(labels_tensor, os.path.join(PROCESSED_DIR, 'labels.pt'))

print(f"전처리 및 정규화 완료. 유효 타격 샘플 수: {len(features)}")
print(f"데이터가 {PROCESSED_DIR}에 저장되었으며, 스케일링 변수가 {EXPORT_DIR}에 기록되었습니다.")

데이터 전처리를 시작합니다...
전처리 및 정규화 완료. 유효 타격 샘플 수: 1975
데이터가 /content/drive/MyDrive/point-6/ai_model/data/processed에 저장되었으며, 스케일링 변수가 /content/drive/MyDrive/point-6/ai_model/exported_models에 기록되었습니다.
